[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P16/P16-2_Etude_DeuxMasses.ipynb)

# Étude IV : système à deux masses

Étude de synthèse de la partie IV du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

Cette étude suit le chemin de toute la partie, de la formule au signal : le calcul symbolique du chapitre 10, les valeurs propres du chapitre 11, l'intégration du chapitre 13, le spectre et les filtres du chapitre 16. Une chaîne relie un mur à deux masses par deux ressorts (figure du chapitre) : $m_1 = 5$ kg, $m_2 = 3$ kg, $k_1 = 2\times 10^6$ N/m et $k_2 = 1\times 10^6$ N/m. Le bureau d'études veut les fréquences propres, la réponse de la première masse écartée de $1$ mm puis lâchée, et la preuve que le spectre de cette réponse retrouve les fréquences calculées. Le résultat attendu est donc un couple de fréquences propres, obtenu par trois chemins qui doivent s'accorder : la formule en lettres, le calcul numérique et la mesure sur le signal. Aucun jeu de données : le signal est simulé, comme un essai.

**Système à deux masses.** Avec $u = (x_1, x_2)$ : $M\,\ddot u + K\,u = 0$, où $M = \mathrm{diag}(m_1, m_2)$ et $K = \begin{pmatrix} k_1 + k_2 & -k_2 \\ -k_2 & k_2 \end{pmatrix}$. Un mode propre est $u = v\cos(\omega t)$ : $(K - \lambda M)\,v = 0$ avec $\lambda = \omega^2$. Il existe si $\det(K - \lambda M) = 0$, polynôme de degré $2$ en $\lambda$ ; $f = \sqrt{\lambda}/(2\pi)$.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("scipy", version("scipy"))
print("sympy", version("sympy"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [ ]:
import math
import numpy as np
import scipy.linalg
import sympy as sp
from scipy.integrate import solve_ivp
from scipy.signal import butter, find_peaks, sosfiltfilt

## Étape 1 : équation caractéristique

Équation caractéristique. M u'' + K u = 0 avec u = v cos(w t) donne (K - lambda M) v = 0, lambda = w^2 ; un mode existe quand det(K - lambda M) = 0.

Les contrôles de la cellule lisent les variables `lam`, `polynome`, `racines` : gardez ces noms.

In [ ]:
# Votre code :
assert len(racines) == 2 and sp.degree(polynome, lam) == 2, "deux modes pour deux masses"
print("Étape 1 : contrôles passés")

## Étape 2 : fréquences numériques

Les mêmes fréquences par un calcul numérique de valeurs propres.

Les contrôles de la cellule lisent les variables `formes`, `freq_modes`, `masses` : gardez ces noms.

In [ ]:
# Votre code :
assert (
    np.allclose(freq_modes, freq_sympy, atol=1e-6)
), "le calcul numérique redonne le calcul en lettres"
assert (
    np.allclose(formes.T @ masses @ formes, np.eye(2))
), "formes propres orthonormées pour la matrice de masse"
print("Étape 2 : contrôles passés")

## Étape 3 : réponse temporelle

Réponse temporelle. Amortissement proportionnel à la raideur, C = beta K, avec beta = 2e-5 s : le taux d'amortissement du mode de pulsation w vaut beta w / 2, soit moins de 1 %.

Les contrôles de la cellule lisent les variables `N`, `solution`, `x1_mm` : gardez ces noms.

In [ ]:
beta = 2e-5
# Votre code :
assert solution.success and solution.y.shape == (4, N), "quatre variables d'état, 4 096 instants"
assert (
    abs(x1_mm[0] - 1.0) < 1e-12 and np.max(np.abs(x1_mm)) <= 1.0 + 1e-9
), "l'amplitude ne dépasse pas le départ"
assert abs(x1_mm[-1]) < 1e-3, "au bout de quatre secondes, la réponse s'est éteinte"
print("Étape 3 : contrôles passés")

## Étape 4 : spectre et fréquences propres

Le spectre de la réponse. Chaque mode donne un pic, à la résolution du spectre près.

Les contrôles de la cellule lisent les variables `f_pics`, `indices`, `resolution`, `spectre` : gardez ces noms.

In [ ]:
# Votre code :
assert resolution == 0.25 and len(f_pics) == 2, "deux pics, comme deux masses"
assert (
    np.all(np.abs(f_pics - freq_modes) <= resolution / 2 + 1e-9)
), "chaque pic est à une demi-résolution du mode"
assert (
    spectre[indices[0]] > spectre[indices[1]]
), "le premier pic est plus haut : le mode rapide s'éteint plus vite (le second mode porte pourtant la plus grande part de l'écart initial)"
print("Étape 4 : contrôles passés")

## Étape 5 : filtrage passe-bas

Un passe-bas à 80 Hz ne garde que le premier mode.

Les contrôles de la cellule lisent les variables `i_pics_filtres`, `reste_second` : gardez ces noms.

In [ ]:
# Votre code :
assert (
    len(i_pics_filtres) == 1 and abs(freq[i_pics_filtres[0]] - freq_modes[0]) <= resolution / 2 + 1e-9
), "un seul pic"
assert reste_second < 0.05, "le second mode est ramené à moins de 5 % de sa valeur"
# Votre code :
print("Étape 5 : contrôles passés")